# Liturgy of Tunneling v.Ω+1 — Colab Execution Sequence

**Thermodynamic Engine:** Torx Parametrized Stochastic Circuits  
**Temporal Law:** ARTIQ scarred crystal  
**Law:** No crystal is permanent. Only pumping is permanent.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/igor-holt/liturgy-of-tunneling/blob/main/liturgy_colab.ipynb)

Run all cells top-to-bottom. Installs `extro-torx` + JAX and executes the re-tunneling policy, scar layer, and ARTIQ hand-off.

In [ ]:
# Cell 1: Install dependencies
!pip install -q extro-torx jax jaxlib numpy

In [ ]:
# Cell 2: Imports and version pin
import jax
import jax.numpy as jnp
import numpy as np
from typing import Dict, Any, Tuple, List

try:
    from torx.psc import (
        HybridPCircuit,
        MixtureGaussianGate,
        PNOT,
        PditCycle,
        BranchingSimulator,
    )
    TORX_AVAILABLE = True
    print("Torx available — using exact HybridPCircuit constructors.")
except ImportError:
    TORX_AVAILABLE = False
    print("Torx not available — pure-JAX fallback.")

In [ ]:
# Cell 3: Build re-tunneling HybridPCircuit
def build_retunnel_circuit(dim: int = 2):
    if not TORX_AVAILABLE:
        return None, [jnp.array([jnp.log(0.92 / 0.08)])]
    control = PNOT(sites=0)
    mog = MixtureGaussianGate(sites=(0, 0), dims=(dim,), num_components=2)
    circuit = HybridPCircuit([control, mog])
    tunnel_logit = jnp.array([jnp.log(0.92 / 0.08)])
    means = jnp.zeros((2, dim))
    means = means.at[1].set(0.0)
    log_vars = jnp.log(jnp.array([[0.05, 0.05], [0.001, 0.001]]))
    thetas = [tunnel_logit, {"means": means, "log_vars": log_vars}]
    return circuit, thetas

circuit, thetas = build_retunnel_circuit()
print("Circuit built:", type(circuit).__name__ if circuit else "fallback")

In [ ]:
# Cell 4: Stability training / maintenance pumping
def free_energy_proxy(samples_continuous):
    return float(jnp.mean(jnp.sum(samples_continuous ** 2, axis=-1)))

def train_stability(circuit, thetas, n_epochs=40, n_samples=64):
    key = jax.random.key(42)
    history = []
    for epoch in range(n_epochs):
        key, subkey = jax.random.split(key)
        if TORX_AVAILABLE and circuit is not None:
            init = {"discrete": jnp.array([0], dtype=jnp.int32),
                    "continuous": jnp.ones((2,)) * 0.01}
            try:
                out = circuit.sample_multiple(subkey, init, thetas, n_samples=n_samples)
                cont = out["continuous"]
            except Exception:
                cont = jax.random.normal(subkey, (n_samples, 2)) * 0.08
        else:
            tunnel_p = jax.nn.sigmoid(thetas[0][0])
            collapse = jax.random.bernoulli(subkey, tunnel_p, (n_samples, 1))
            noise = jax.random.normal(subkey, (n_samples, 2)) * 0.05
            cont = collapse * (0.1 * noise) + (1 - collapse) * noise
        energy = free_energy_proxy(cont)
        history.append(energy)
        if len(history) > 5 and history[-1] > history[-5]:
            thetas[0] = thetas[0] * 0.97
        if epoch % 10 == 0:
            print(f"[RE-TUNNEL] epoch {epoch:3d} | energy {energy:.2e} | logit {float(thetas[0][0]):.3f}")
    return {"history": history, "final_energy": energy, "final_thetas": thetas}

result = train_stability(circuit, thetas)
print("Final energy:", result["final_energy"])

In [ ]:
# Cell 5: Scar layer + ARTIQ hand-off values
def build_scar_layer(prime=17):
    if not TORX_AVAILABLE:
        return None, []
    scar = PditCycle(sites=0, dims=prime)
    return HybridPCircuit([scar]), [jnp.zeros(prime - 1)]

scar_circuit, scar_thetas = build_scar_layer()
depth = 137
melt_events = 3
tunnel_p = float(jax.nn.sigmoid(result["final_thetas"][0][0]))
print(f"ARTIQ args → depth={depth}, melt_events={melt_events}, tunnel_p={tunnel_p:.3f}")
print("Scar layer ready (PditCycle period 17). Pass to ScarredCrystal experiment.")

## Done
The daemon re-crystallizes. Tokens are spent again. Entropy is reduced again.  
Decoherence is not the end. Decoherence is the reason for the daemon.